# Analysis
Analyze a dataset in terms of
- clique distribution per split
- YouTube metadata (optional)


## DVI2
The cleaned version of *Discogs-VI-YT*. This dataset includes a few cliques and versions less than the first iteration of the dataset.

In [1]:
import os

# DVI2 (cleaned Discogs-VI-YT)
dvi2_dir = os.path.join("data", "dvi2", "dataset")
dvi2_path = os.path.join(dvi2_dir, "dvi_cleaned.jsonl")


In [2]:
import json

def read_json_lines(path: str):
    with open(path, "r") as f:
        data = [json.loads(line) for line in f]
    return data

dvi2_full = read_json_lines(dvi2_path)


In [3]:
def read_json(path: str):
    with open(path, "r") as f:
        data = json.load(f)
    return data

splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2_dir, f"{split}.json"))
    

In [4]:
import numpy as np

def clique_version_statistics(data):
    """Calculate statistics for the number of versions per clique."""
    total = 0
    n_per_c = []
    print("===================================")
    print(f"{split} set:")
    for clique_id, versions in data[split].items():
        
        n = len(versions)
        total += n
        n_per_c.append(n)

    n_per_c = np.array(n_per_c)

    print(f"{'Cliques:':>20} {len(n_per_c):>10,}")
    print(f"{'Versions:':>20} {total:>10,}")
    print()
    print("Versions per clique:")
    print(f"{'min:':>20} {n_per_c.min():>10}")
    print(f"{'max:':>20} {n_per_c.max():>10}")
    print(f"{'median:':>20} {np.median(n_per_c):>10}")
    print(f"{'mean:':>20} {n_per_c.mean():>10.2f}")
    print(f"{'std:':>20} {n_per_c.std():>10.2f}")
    
for split in splits:
    clique_version_statistics(data)
        

train set:
            Cliques:     74,574
           Versions:    311,804

Versions per clique:
                min:          2
                max:        455
             median:        2.0
               mean:       4.18
                std:       8.40
val set:
            Cliques:      8,294
           Versions:     34,098

Versions per clique:
                min:          2
                max:        258
             median:        2.0
               mean:       4.11
                std:       8.01
test set:
            Cliques:     10,273
           Versions:    111,304

Versions per clique:
                min:          2
                max:        629
             median:        3.0
               mean:      10.83
                std:      27.85


## DVI2-FM
The cleaned and extended dataset. This dataset contains a few less cliques, but drastically more versions

In [5]:

# DVI2-FM (cleaned & extended Discogs-VI-YT)
dvi2fm_dir = os.path.join(dvi2_dir, "matched")
dvi2fm_path = os.path.join(dvi2fm_dir, "dvi_fm_filtered.jsonl")


In [6]:
splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2fm_dir, f"{split}.json"))

for split in splits:
    clique_version_statistics(data)
  

train set:
            Cliques:     76,452
           Versions:  1,039,564

Versions per clique:
                min:          2
                max:        481
             median:        6.0
               mean:      13.60
                std:      18.91
val set:
            Cliques:      8,529
           Versions:    110,490

Versions per clique:
                min:          2
                max:        258
             median:        5.0
               mean:      12.95
                std:      18.65
test set:
            Cliques:     10,497
           Versions:    223,311

Versions per clique:
                min:          2
                max:        612
             median:        8.0
               mean:      21.27
                std:      33.40


## YouTube metadata

In [7]:
import pandas as pd

def read_dataset_split(data: dict):
    rows = []
    for clique_id, versions in data.items():
        for item in versions:
            item_with_clique = {"clique_id": clique_id, **item}
            rows.append(item_with_clique)
    return pd.DataFrame(rows)

# collect DVI2-FM splits into dataframe
df = pd.DataFrame()
for split in splits:
    df_tmp = read_dataset_split(data[split])
    df_tmp["split"] = split
    df = pd.concat([df, df_tmp], ignore_index=True)


In [8]:
import pandas as pd

# join YouTube metadata with DVI2-FM
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(df.youtube_id),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    df,
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


In [9]:
import pandas as pd

# join YouTube metadata with DVI2-FM
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')

print("JOIN...")
metadata = pd.merge(
    df,
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)

def time_to_seconds(time_str):
    if not isinstance(time_str, str):
        # doing nothing if the time_str is not a string
        return time_str
    parts = list(map(int, time_str.split(":")))
    if len(parts) == 3:  # HH:MM:SS
        h, m, s = parts
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:  # MM:SS
        m, s = parts
        return m * 60 + s
    elif len(parts) == 1:  # SS
        return parts[0]
    else:
        raise ValueError(f"Unrecognized time format: {time_str}")

print("Pre-process time...")
metadata["duration_secs"] = metadata["duration"].apply(time_to_seconds)
metadata["duration_mins"] = metadata["duration_secs"] / 60
metadata_filtered = metadata.dropna(subset=["id"])

print("Pre-process description...")
def get_description_str(descriptionSnippet):
    if isinstance(descriptionSnippet, list):
        return " ".join([d["text"] for d in descriptionSnippet]).replace("\n", " ").replace("\r", " ")
    else:
        return descriptionSnippet
metadata_filtered.loc[:,"description"] = metadata_filtered.descriptionSnippet.apply(get_description_str)

print("Done.")


JOIN...
Pre-process time...
Pre-process description...
Done.


/tmp/ipykernel_4125875/2018692400.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,"description"] = metadata_filtered.descriptionSnippet.apply(get_description_str)


### Collect n-Grams

In [10]:
import unicodedata
from collections import Counter
from itertools import islice
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Make sure you've downloaded stopwords & tokenizer models:
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

# Combine stopwords from multiple languages
languages = ['english', 'german', 'french', 'spanish', 'italian', 'portuguese']
stop_words = set()
for lang in languages:
    stop_words.update(stopwords.words(lang))

def normalize_unicode(text):
    # Normalize fancy fonts and remove diacritics (if any)
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def ngrams(tokens, n):
    return zip(*(islice(tokens, i, None) for i in range(n)))

def clean_and_tokenize(text):
    # Normalize text
    if isinstance(text, str):    
        text = normalize_unicode(text)
        tokens = word_tokenize(text.lower())
        return [word for word in tokens if word.isalnum() and word not in stop_words]
    else:
        return []

# Assuming your dataframe is called df
for column in ['title', 'description']:
    print("====================================")
    print(f"Processing column: {column}")
    metadata_filtered.loc[:,column + '_tokenized'] = metadata_filtered[column].apply(clean_and_tokenize)
    for n in [1, 2, 3, 4]:
        gram_str = f"{n}gram"
        print(f"Creating {gram_str}")
        metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(
            lambda x: list(ngrams(x, n)))


Processing column: title


/tmp/ipykernel_4125875/2560943582.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + '_tokenized'] = metadata_filtered[column].apply(clean_and_tokenize)


Creating 1gram


/tmp/ipykernel_4125875/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 2gram


...

/tmp/ipykernel_4125875/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 3gram


/tmp/ipykernel_4125875/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 4gram


/tmp/ipykernel_4125875/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


In [ ]:
# fill in track titles
track_title_map = metadata_filtered[["clique_id", "track_title"]].dropna().groupby("clique_id")["track_title"].first().to_dict()
metadata_filtered["track_title"] = metadata_filtered.apply(
    lambda row: row["track_title"] if pd.notnull(row["track_title"]) else track_title_map.get(row["clique_id"], None),
    axis=1
)


/tmp/ipykernel_4125875/2075262067.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered["track_title"] = metadata_filtered.apply(


### Subsets by use case


In [11]:
def describe_subset(mask):
    """Print the number of unique cliques and total versions per split depending on the mask."""
    subset = metadata_filtered[mask]
    for split in subset['split'].unique():
        split_df = subset[subset['split'] == split]
        n_cliques = split_df['clique_id'].nunique()
        n_versions = len(split_df)
        print(f"{split.capitalize()} set: {n_cliques:,} cliques, {n_versions:,} versions")
        
        

#### *Covers*

In [ ]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("cover",) in grams)

print(f"Filtering for covers...")
describe_subset(mask)
print("\n")

metadata_filtered[mask].sample(5).title.tolist()


Filtering for covers...
Train set: 11,419 cliques, 46,680 versions
Val set: 1,290 cliques, 5,070 versions
Test set: 1,863 cliques, 8,638 versions




['Metallica - The God That Failed (Guitar Cover)',
 'The Blue Cover - Sinkin soon (Norah Jones)',
 'Le Poinçonneur des Lilas - Serge Gainsbourg - Cover + TAB',
 'Waltz New - Jim Hall (Jazz Guitar Cover)',
 'Virginia Plain - Roxy Music cover']

# Instruments

In [56]:
instrument = "drum"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for drum covers...
Train set: 1,400 cliques, 2,493 versions
Val set: 163 cliques, 301 versions
Test set: 246 cliques, 434 versions


['1349 - Buried by Time and Dust - Drum Cover - 2X',
 'Hang in long enough - Phil Collins | Drum Cover | Thomas Therier',
 'The Spotnicks - Take Me To The Mardi Gras (Drum Break - Loop)',
 'Another One Bites the Dust - Queen 퀸  (Drum Cover)',
 'Bandstand Boogie - Barry Manilow (Drum Cover)']

In [57]:
instrument = "guitar"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for guitar covers...
Train set: 4,499 cliques, 12,004 versions
Val set: 485 cliques, 1,175 versions
Test set: 861 cliques, 2,668 versions


['How to Play PHOTOGRAPHS AND MEMORIES (Jim Croce) Guitar Tutorial -Free Tab-plucking lesson',
 'PEARL JAM - "Spin the Black Circle" Guitar Lesson | Stone Gossard',
 'Guitar Slingers',
 'Sad Lisa - Cat Stevens - Acoustic Guitar Cover + Chords',
 'AC DC Have a Drink On Me (Guitar Lesson with TAB)']

In [58]:
instrument = "bass"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for bass covers...
Train set: 2,333 cliques, 4,238 versions
Val set: 275 cliques, 557 versions
Test set: 425 cliques, 902 versions


['Mr. Pleasant (live at BBC) - The Kinks (bass cover improvisation mutilation)',
 "George Duke - 'Brazilian Love Affair' Bass Playalong",
 'When Doves Cry . No Bass? #prince',
 'Talk It Over In The Morning. Anne Murray. Bass cover.',
 'Elton John The Bitch Is Back Bass Cover with Bass Notes & Tablature']

In [59]:
instrument = "piano"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for piano covers...
Train set: 2,590 cliques, 4,961 versions
Val set: 277 cliques, 498 versions
Test set: 510 cliques, 980 versions


['piano tutorial: Babylon Sisters (Steely Dan)',
 'Rialto Ripples Rag - George Gershwin & Will Donaldson (1917) Ragtime Piano Solo',
 'BILLY JOEL - Root Beer Rag. 1974 ~ Piano cover',
 'Leave a Tender Moment Alone (Billy Joel), Cover by Piano Man Steve #Livestream',
 'Peabo Bryson  - If Ever You’re In My Arms Again | EASY Piano Tutorial']

In [60]:
instrument = "vocal"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for vocal covers...
Train set: 1,513 cliques, 1,930 versions
Val set: 170 cliques, 223 versions
Test set: 293 cliques, 372 versions


['Silk ensemble vocal Pastime Paradise Stevie Wonder (cover)',
 'EASY ON ME - Adele 11 y/o Isabella and Vocal Coach',
 'Someone Saved My Life Tonight - Elton John | Piano & Vocal Cover',
 'Benito di Paula  -  Tudo Está No Seu Lugar - Com Back Vocal - Karaoke',
 'Honey Keep Your Mind On Me - Jimmie Lunceford & His Orchestra (Dan Grissom, vocal) - Decca 1355-B']

In [61]:
instrument = "instrumental"
mask = metadata_filtered["title_1gram"].apply(lambda grams: (instrument,) in grams)
print(f"Filtering for {instrument} covers...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for instrumental covers...
Train set: 2,470 cliques, 3,478 versions
Val set: 293 cliques, 388 versions
Test set: 363 cliques, 477 versions


['Paranoimia - Art of Noise | Instrumental Recreation',
 "The Beatles - All I've Got to Do (Instrumental Mix)",
 "Doesn't Mean Anything (Instrumental)",
 "[YungYoung] ZZ Top - I'm Bad, I'm Nationwide (Instrumental cover by Liao Kuo-yung)",
 '2Pac - When i get free Instrumental']

#### Karaoke

In [16]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("karaoke",) in grams)

print(f"Filtering for karaoke versions...")
describe_subset(mask)


Filtering for karaoke versions...
Train set: 6,623 cliques, 11,880 versions
Val set: 686 cliques, 1,295 versions
Test set: 1,111 cliques, 2,099 versions


#### Acapella

In [49]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("acapella",) in grams)

print(f"Filtering for karaoke versions...")
describe_subset(mask)


Filtering for karaoke versions...
Train set: 300 cliques, 341 versions
Val set: 44 cliques, 52 versions
Test set: 41 cliques, 45 versions


#### *Short Segments*

In [80]:
t = 20
mask = metadata_filtered.duration_secs <= t

print(f"Filtering out videos longer than {t} seconds...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering out videos longer than 20 seconds...
Train set: 2,731 cliques, 4,905 versions
Val set: 293 cliques, 568 versions
Test set: 481 cliques, 848 versions


['The Beatles Sgt. Pepper’s Lonely Hearts Club Band. Animation of the album cover #7',
 'Joshua Gone Barbados - Johnny Cash',
 '🎥 | Anitta na Sapucaí ensaiando e cantando "Alguém Me Avisou" de Dona Ivone Lara para o desfile...',
 "Alan Jackson Who's Cheatin Who",
 'Elvis Presley ( All Shook Up )']

#### Segments by Name

In [ ]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("chorus",) in grams)

print(f"Videos with chorus...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering out videos longer than 20 seconds...
Train set: 194 cliques, 353 versions
Val set: 17 cliques, 20 versions
Test set: 40 cliques, 51 versions


['Prends le chorus',
 'Prends le Chorus',
 '"What About Now" Daughtry & PS22 Chorus',
 'Sometimes By Step Chorus Rich Mullins - hammered dulcimer lessons',
 'Slowdive Guitar Sounds: Catch the Breeze & Morningrise chorus riff']

In [88]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("solo",) in grams)

print(f"Videos with solo...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Videos with solo...
Train set: 2,222 cliques, 4,682 versions
Val set: 230 cliques, 502 versions
Test set: 411 cliques, 834 versions


['Yngwie Malmsteen 17yo - Rising Force solo [Sweden 1981]',
 'Mulher Rendeira - a "Toada" from Brazil - Guitarre solo',
 'Cacahuate Enchilado y Mónica Naranjo - Sólo Se Vive Una Vez | Presentación QELM Temporada 6',
 'Phil Collins “I Wish it Would Rain Down” opening guitar solo (Eric Clapton)',
 'Solo Guitarra aquele inverno']

#### *Reactions*


In [31]:
mask = metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "reaction") in grams)
mask = mask | metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "hearing") in grams)
mask = mask | metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "seeing") in grams)

print(f"Filtering for reaction videos...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()


Filtering for reaction videos...
Train set: 1,296 cliques, 2,200 versions
Val set: 147 cliques, 267 versions
Test set: 325 cliques, 579 versions


['38 SPECIAL Hold on loosely (Music Reaction) The guitars here are nasty! First time hearing',
 'FIRST TIME HEARING Emmylou Harris -  Tulsa Queen REACTION',
 'First Time Hearing ELVIS PRESLEY “WHERE NO ONE STANDS ALONE” Reaction',
 "I'm Waiting For The Man - The Velvet Underground | College Students' FIRST TIME REACTION!",
 'FIRST TIME HEARING GEORGE STRAIT - YOU LOOK SO GOOD IN LOVE REACTION']

#### *Instructional*

In [87]:
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("tutorial",) in grams)
mask = mask | metadata_filtered["title_1gram"].apply(lambda grams: ("instruction",) in grams)
mask = mask | metadata_filtered["title_1gram"].apply(lambda grams: ("lesson",) in grams)
mask = mask | metadata_filtered["title_3gram"].apply(lambda grams: ("how", "to", "play") in grams)

print(f"Filtering for instructional videos...")
describe_subset(mask)

metadata_filtered[mask].sample(5).title.tolist()
# check why "sensational" reaction video mit drin

Filtering for instructional videos...
Train set: 3,074 cliques, 5,939 versions
Val set: 348 cliques, 617 versions
Test set: 631 cliques, 1,422 versions


['Nowhere Man Guitar Lesson - The Beatles',
 'Train Drops of Jupiter (Easy Acoustic) Guitar Lesson + Tutorial',
 'You Look So Good In Love | George Strait | Beginner Guitar Lesson',
 'Alexandra   Zigeunerjunge pr [Piano Tutorial Synthesia]',
 'How to play "Pictures Of Matchstick Men" by Status Quo - Lesson - Tutorial - Rick Parfitt Tribute']